# Known discrete state potential

This notebook adds a known potential `U_xi` on the encoded state label. It then checks two things:

1. whether a MetaTally-style bias discovers states faster than an independent torsion bias;
2. whether a frozen-bias production run can recover the imposed state free-energy differences.


In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np

import metatally as mt


In [ ]:
rng = np.random.default_rng(4)

n_variables = 2
radix = 3
barrier = 1.5
step_size = np.pi / 10

# Build a known discrete potential over xi.
space0 = mt.TorsionChain(n_variables=n_variables, radix=radix).state_space()
uxi = rng.normal(scale=0.8, size=space0.n_states)
uxi -= uxi[0]

model = mt.TorsionChain(
    n_variables=n_variables,
    radix=radix,
    barrier=barrier,
    coupling=0.0,
    state_potential=uxi,
)
space = model.state_space()

true_dF = model.state_free_energy_differences(reference=0)
print('n_states =', space.n_states)
print('true dF =', np.round(true_dF, 3))


In [ ]:
biases = {
    'plain': mt.NoBias(),
    'independent V(phi)': mt.IndependentTorsionBias.from_space(
        space, height=0.01, sigma=0.2, n_grid=96
    ),
    'V(xi) + V(phi)': mt.CompositeBias([
        mt.DiscreteStateBias.from_space(space, height=0.05),
        mt.IndependentTorsionBias.from_space(space, height=0.01, sigma=0.2, n_grid=96),
    ]),
}

samplers = {}
for name, bias in biases.items():
    sampler = mt.MetropolisSampler(
        model=model,
        state_space=space,
        bias=bias,
        initial=model.initial_state(),
        step_size=step_size,
        beta=1.0,
        seed=10,
    )
    sampler.run(5_000)
    samplers[name] = sampler

for name, sampler in sorted(samplers.items(), key=lambda item: item[1].coverage, reverse=True):
    print(
        f'{name:20s} n_visited={sampler.n_visited:3d} '
        f'coverage={sampler.coverage:.3f} acceptance={sampler.acceptance_rate:.3f}'
    )


In [ ]:
plt.figure()
for name, sampler in samplers.items():
    plt.plot(sampler.steps, sampler.n_unique_by_step / space.n_states, label=name)
plt.xlabel('MC step')
plt.ylabel('fraction of states discovered')
plt.legend()
plt.tight_layout()


In [ ]:
# Freeze the best discovery bias and run fixed-bias production.
adaptive = samplers['V(xi) + V(phi)']
frozen_bias = copy.deepcopy(adaptive.bias)
frozen_bias.freeze()

production = mt.MetropolisSampler(
    model=model,
    state_space=space,
    bias=frozen_bias,
    initial=adaptive.x,
    step_size=step_size,
    beta=adaptive.beta,
    seed=11,
)
production.run(20_000)

probs = mt.reweighted_state_probabilities(
    production.states,
    production.bias_values,
    beta=production.beta,
    n_states=space.n_states,
)
estimated_dF = mt.reweighted_free_energy_differences(
    production.states,
    production.bias_values,
    beta=production.beta,
    n_states=space.n_states,
    reference=0,
)

print(mt.reweighting_diagnostics(production.bias_values, beta=production.beta))
print('estimated probabilities:', np.round(probs, 3))


In [ ]:
plt.figure()
plt.plot(true_dF, estimated_dF, 'o')
lo = min(np.nanmin(true_dF), np.nanmin(estimated_dF))
hi = max(np.nanmax(true_dF), np.nanmax(estimated_dF))
plt.plot([lo, hi], [lo, hi], 'k--', lw=1)
plt.xlabel('known $U_\xi - U_0$')
plt.ylabel('estimated $\Delta F$')
plt.tight_layout()


In [ ]:
for xi, (true, est) in enumerate(zip(true_dF, estimated_dF)):
    print(f'{xi:2d}  true={true:8.3f}  estimated={est:8.3f}')
